# Bass Diffusion Popularity Function for Telecom Bundles

* in this notebook we will define the popularity function for telecom bundles based on our popularity function from ***popularity_monthly.ipynb*** notebook, and the power with Bass Diffusion Model.

* The Bass diffusion model is a mathematical model that describes how new products get adopted in a population over time. **It’s widely used in marketing to forecast sales of new product**s, **especially when historical data is limited or non-existent.**


* [Bass Diffusion Model](https://www.pymc-marketing.io/en/stable/notebooks/bass/bass_example.html) Explained!

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.mixture import GaussianMixture
from tqdm import tqdm
from scipy.optimize import curve_fit


import sys
from pathlib import Path

repo_root = Path("../../../mintel")
if repo_root not in map(Path, sys.path):
    sys.path.insert(0, str(repo_root))

In [2]:
DATASET_FOLDER: str = r"..\..\..\datasets"
MONTHLY_BUNDLES_DATA_PATH: str = rf"{DATASET_FOLDER}\monthly_base_bundles_info_202501_202510.csv"


data = pd.read_csv(MONTHLY_BUNDLES_DATA_PATH) 
data

,month_number,year_number,bundle_id,bundle_name,bundle_type,validity,price,usage_type,configured_volume,service_class_category,total_duration,total_rev,total_subscriptions,total_sessions,unique_users
0,1,2025,FVX7D840FV3,Maxivoice 90Mins 7 jour@840F,BUNDLE_VOICE,7 Day(s),840,CHARGED,90min,PREPAID,0.0,94717559.86,112759,112759,33989
1,1,2025,36003,Forfait DIY DATA HIGH,BUNDLE_DATA,NaN,NaN,CHARGED,NaN,STAFF,0.0,20950.00,12,12,11
2,1,2025,36848,Zone A and B NDAKO 550GB 30 Jours@26500F,BUNDLE_DATA,30 Day(s),26500,CHARGED,550GB,PREPAID,0.0,2782500.00,105,105,105
3,1,2025,FVX1D120F,"Maxivoice 7,5Mins and 1,5Mins 1 jour@120F",BUNDLE_VOICE,1 Day(s),120,CHARGED,"7,5Mins and 1,5Mins",HYBRID,0.0,480.00,4,4,1
4,1,2025,36010,Forfait DIY COMBO DATA MEDIUM,BUNDLE_DATA,NaN,NaN,CHARGED,NaN,PREPAID,0.0,1567570.00,3481,3497,1443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12307,10,2025,36207,Forfait Classic 7 jours 750MB@2000,BUNDLE_DATA,7 Day(s),2000,CHARGED,750MB,PREPAID,0.0,70000.00,35,35,21
12308,10,2025,36972,MagicNet 20.2GB 30 Jours@13000F,BUNDLE_DATA,30 Day(s),13000,CHARGED,20.2GB,HYBRID,0.0,364000.00,28,28,28
12309,10,2025,35074,Forfait 1 jour 28 MB,BUNDLE_DATA,1 Day(s),130,CHARGED,28 MB,PREPAID,0.0,1604070.00,13426,13473,8878
12310,10,2025,80141,Forfait Chrono voice 40 Mins 4H@200F,BUNDLE_VOICE,4 Hours,200,CHARGED,40Min,PREPAID,0.0,4375517.00,22162,22795,10990


In [3]:
# same data preprocessing in the popularity_monthly.ipynb
def season(month_number):
    if 3 <= month_number <= 5:
        return "Spring"
    elif 6 <= month_number <= 8:
        return "Summer"
    elif 9 <= month_number <= 11:
        return "Autumn"  
    elif month_number == 12 or 1 <= month_number <= 2:
        return "Winter"
    else:
        return "UNK"

data['season'] = data['month_number'].apply(season)
# Extract Configured Volumes 'configured_volume'
def extract_conf_volume_values(volume_str):
    """
    Extract MB, minutes, and SMS from configured_volume string.
    Handles: GB (→ MB), MB, Minutes, SMS, commas, combinations, addition formats, etc.
    """
    if pd.isna(volume_str) or volume_str is None:
        return {'mb': None, 'min': None, 'sms': None}
    
    original_str = str(volume_str)
    if original_str.lower() in ['unlimited', 'illimité', 'illimite']:
        return {'mb': None, 'min': None, 'sms': None}
    
    volume_str = original_str.replace(',', '.')
    volume_str_upper = volume_str.upper()
    
    mb_value = None
    min_value = None
    sms_value = None
    
    # extract GB and convert to MB
    gb_pattern = r'(\d+(?:\.\d+)?)\s*G(?:B)?\b'
    gb_matches = re.findall(gb_pattern, volume_str_upper)
    if gb_matches:
        mb_value = sum(float(gb) for gb in gb_matches) * 1024
    
    # extract MB
    mb_pattern = r'(\d+(?:\.\d+)?)\s*MB\b'
    mb_matches = re.findall(mb_pattern, volume_str_upper)
    if mb_matches:
        total_mb = sum(float(mb) for mb in mb_matches)
        mb_value = total_mb if mb_value is None else mb_value + total_mb
    
    # extract minutes
    min_pattern = r'(\d+(?:\.\d+)?)\s*MINS?\b'
    min_matches = re.findall(min_pattern, volume_str_upper)
    add_pattern_min = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)\s*MINS?\b'
    add_match_min = re.search(add_pattern_min, volume_str_upper)
    
    if add_match_min:
        min_value = float(add_match_min.group(1)) + float(add_match_min.group(2))
    elif min_matches:
        min_value = sum(float(m) for m in min_matches)
    
    # extract SMS
    sms_pattern = r'(\d+(?:\.\d+)?)\s*SMS\b'
    sms_matches = re.findall(sms_pattern, volume_str_upper)
    if sms_matches:
        sms_value = sum(float(sms) for sms in sms_matches)
    
    # handle "/" formats like "200/40 mins"
    slash_pattern = r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)'
    slash_match = re.search(slash_pattern, volume_str)
    if slash_match and 'MIN' in volume_str_upper:
        if min_value is None:
            min_value = float(slash_match.group(2))
        if mb_value is None:
            mb_value = float(slash_match.group(1))
    
    # handle seconds
    sec_pattern = r'(\d+(?:\.\d+)?)\s*SEC\b'
    sec_matches = re.findall(sec_pattern, volume_str_upper)
    if sec_matches:
        sec_as_min = sum(float(sec) for sec in sec_matches) / 60.0
        min_value = sec_as_min if min_value is None else min_value + sec_as_min
    
    return {'mb': mb_value, 'min': min_value, 'sms': sms_value}


volume_extracted = data['configured_volume'].apply(extract_conf_volume_values)
data['configured_volume_mb'] = volume_extracted.apply(lambda x: x['mb'] if x['mb'] is not None else 0.0)
data['configured_volume_min'] = volume_extracted.apply(lambda x: x['min'] if x['min'] is not None else 0.0)
data['configured_volume_sms'] = volume_extracted.apply(lambda x: x['sms'] if x['sms'] is not None else 0.0)
data.drop(columns=['configured_volume'], inplace=True)
def clean_price(val) -> float:
        if pd.isna(val):
            return None
        val = str(val).replace("F", "").replace(",", "")
        try:
            return float(val)
        except:
            return None
        
data["price"] = data["price"].apply(clean_price)
data['clean_validity'] = np.where(
    data['validity'].str.strip().str.lower().isin(['null', '']) | data['validity'].isna(),
    None,
    data['validity']
        .str.strip()
        .str.lower()
        .str.replace(r'\(s\)', '', regex=True)         # remove (s)
        .str.replace(r'\bday\b', 'days', regex=True)  # day → days
        .str.replace(r'\bmonth\b', 'months', regex=True)  # month → months
)
data.drop(columns=['validity'], inplace=True)
data = data.rename(columns={"clean_validity": "validity"})

def parse_validity_to_hours(val):
    """
    Convert variants like '7 days', '4 hours', '3 months', '1  days', None -> hours (float)
    Uses 30 days per month convention for telecom.
    Returns np.nan if cannot parse.
    """
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        # unexpected numeric — treat as days
        return float(val) * 24.0
    s = str(val).strip().lower()
    # correct double spaces
    s = re.sub(r"\s+", " ", s)
    # find number
    m = re.match(r"^(\d+(\.\d+)?)\s*(day|days|hour|hours|month|months)?", s)
    if not m:
        return np.nan
    num = float(m.group(1))
    unit = m.group(3) or "days"
    if "hour" in unit:
        return num
    if "day" in unit:
        return num * 24.0
    if "month" in unit:
        return num * 30.0 * 24.0
    # fallback assume days
    return num * 24.0

data['validity_hours'] = data['validity'].apply(parse_validity_to_hours)
data = data[~data["bundle_name"].str.contains("DIY", case=False, na=False)]
data.drop(columns=['total_duration'], inplace=True)
data.dropna(subset=['bundle_name'], inplace=True)
data['service_class_category'] = data['service_class_category'].fillna(data['service_class_category'].mode()[0])
data['validity'] = data['validity'].fillna('NO_VALIDITY')
data['validity_hours'] = data['validity_hours'].fillna(0)
def extract_price(text):
    if pd.isna(text):
        return None
    match = re.search(r'@(\d+)\s*F', str(text))
    if match:
        return float(match.group(1))
    return None

# try to extract price from bundle_name for all rows with missing price
mask_missing_price = data['price'].isna()
data.loc[mask_missing_price, 'price_extracted'] = \
    data.loc[mask_missing_price, 'bundle_name'].apply(extract_price)

# fill price where extraction was successful
data['price'] = data['price'].fillna(data['price_extracted'])


# 4. FREE bundles (@F)
free_mask = data['bundle_name'].str.contains('@F', na=False)
data.loc[free_mask, 'price'] = 0

# STAFF bundles
staff_mask = data['bundle_name'].str.contains('STAFF', na=False, case=False)
data.loc[staff_mask, 'price'] = 0

# VAS services and system events
system_names = ['SDP VAS', 'MTN ME2U', 'MTN Xtratime', 
                'MTN Call Me Back', 'MTN Tv']

system_mask = data['bundle_name'].isin(system_names)
data.loc[system_mask, 'price'] = 0

# Anything still missing → truly unknown, set to zero but mark category
still_missing = data['price'].isna()
data.loc[still_missing, 'price'] = 0

data.drop(columns=['price_extracted'], inplace=True)

In [4]:
data

,month_number,year_number,bundle_id,bundle_name,bundle_type,price,usage_type,service_class_category,total_rev,total_subscriptions,total_sessions,unique_users,season,configured_volume_mb,configured_volume_min,configured_volume_sms,validity,validity_hours
0,1,2025,FVX7D840FV3,Maxivoice 90Mins 7 jour@840F,BUNDLE_VOICE,840.0,CHARGED,PREPAID,94717559.86,112759,112759,33989,Winter,0.0,90.0,0.0,7 days,168.0
2,1,2025,36848,Zone A and B NDAKO 550GB 30 Jours@26500F,BUNDLE_DATA,26500.0,CHARGED,PREPAID,2782500.00,105,105,105,Winter,563200.0,0.0,0.0,30 days,720.0
3,1,2025,FVX1D120F,"Maxivoice 7,5Mins and 1,5Mins 1 jour@120F",BUNDLE_VOICE,120.0,CHARGED,HYBRID,480.00,4,4,1,Winter,0.0,9.0,0.0,1 days,24.0
5,1,2025,36206,Forfait Classic 7 jours 375MB@1200,BUNDLE_DATA,1200.0,CHARGED,HYBRID,22800.00,19,19,12,Winter,375.0,0.0,0.0,7 days,168.0
6,1,2025,ALLNET10MN1D450F,Forfait MagicVoice tous reseaux 1 jour 10 mins...,BUNDLE_VOICE,450.0,CHARGED,HYBRID,1350.00,3,3,3,Winter,0.0,0.0,0.0,1 days,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12307,10,2025,36207,Forfait Classic 7 jours 750MB@2000,BUNDLE_DATA,2000.0,CHARGED,PREPAID,70000.00,35,35,21,Autumn,750.0,0.0,0.0,7 days,168.0
12308,10,2025,36972,MagicNet 20.2GB 30 Jours@13000F,BUNDLE_DATA,13000.0,CHARGED,HYBRID,364000.00,28,28,28,Autumn,20684.8,0.0,0.0,30 days,720.0
12309,10,2025,35074,Forfait 1 jour 28 MB,BUNDLE_DATA,130.0,CHARGED,PREPAID,1604070.00,13426,13473,8878,Autumn,28.0,0.0,0.0,1 days,24.0
12310,10,2025,80141,Forfait Chrono voice 40 Mins 4H@200F,BUNDLE_VOICE,200.0,CHARGED,PREPAID,4375517.00,22162,22795,10990,Autumn,0.0,40.0,0.0,4 hours,4.0


### Define Bass Diffusion Popularity Functoin

### A)- First well keep the popularity functoin as it was in the previous study.

$$
P^{(e)}_b = \text{total\_subscriptions}_b
$$

$$
R_b = \text{total\_rev}_b
$$


$$
U_b = \text{unique\_users}_b
$$

* Zero-Usage Indicator
$$
A_b = \text{total\_sessions}_b
$$


$$
N^0_b = 
\begin{cases}
1 & \text{if } A_b = 0 \\
0 & \text{otherwise}
\end{cases}
$$


* Diversity Score: Using embedding 𝑒𝑏 and cluster centroid  c𝑏
$$
D_b = 1 - \cos\left(e_b,\, c_b\right)
$$

$$
\hat{D}_b = \frac{D_b - D_{\min}}{D_{\max} - D_{\min} + \varepsilon}
$$

* Final Telecom Popularity Function

$$
\text{Pop}(b) = 
\alpha\,\hat{P}^{(e)}_b
+ \beta\,\hat{R}_b
+ \gamma\,\hat{U}_b
+ \eta\,(1 - N^0_b)
+ \zeta\,\hat{D}_b
$$


$$
\alpha = 0.35,\qquad
\beta = 0.30,\qquad
\gamma = 0.20,\qquad
\eta = 0.10,\qquad
\zeta = 0.0500000000000001
$$



The popularity category of a bundle \(b\) based on the Gaussian Mixture Model (GMM) can be defined as:

$$
\text{Cat}_{\text{GMM}}(b) =
\begin{cases} 
\text{unpopular} & \text{if } \text{Pop}(b) < \frac{\mu_1 + \mu_2}{2} \\[1em]
\text{popular} & \text{if } \frac{\mu_1 + \mu_2}{2} \le \text{Pop}(b) < \frac{\mu_2 + \mu_3}{2} \\[1em]
\text{very popular} & \text{if } \text{Pop}(b) \ge \frac{\mu_2 + \mu_3}{2}
\end{cases}
$$

where:

𝜇1 , 𝜇2 , 𝜇3​ are the means of the three Gaussian components from the GMM.

---------------------------------------------------------------------------------------------------------------------------------

### B)- Introduce the Bass Diffusion Model to our Popularity Funciton 


In [5]:
# A)- this cell to compute the basic popularity function
def norm(col: pd.Series, eps: float = 1e-9) -> pd.Series:
    """
    - Min-max normalize series to [0,1]. Works with constant series safely.
    """
    col = col.astype(float)
    mi = col.min()
    ma = col.max()
    denom = (ma - mi) + eps
    return (col - mi) / denom


def compute_popularity(df, n_components=128, n_clusters=20, eps=1e-9):
    """
    Compute popularity using TF-IDF embeddings 
    """
    df = df.copy()

    df["P_e_norm"] = norm(df["total_subscriptions"])
    df["R_norm"]   = norm(df["total_rev"])
    df["U_norm"]   = norm(df["unique_users"])
    df["A_norm"]   = norm(df["total_sessions"])

    df["N0_norm"]  = df["total_sessions"].apply(lambda x: 1 if x==0 else 0)


    # TF-IDF vectorization
    print("TF-IDF vectorization")
    vectorizer = TfidfVectorizer(
        analyzer="word",
        stop_words="english",
        max_features=5000
    )
    X_tfidf = vectorizer.fit_transform(df["bundle_name"])

    # Dimensionality reduction (SVD acts like LSA embeddings)
    print("Dimensionality reduction")
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_emb = svd.fit_transform(X_tfidf)

    df["embedding"] = list(X_emb)

    print(f"{len(X_emb)} Embdedd.")
    # compute diversity
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X_emb)
    centroids = kmeans.cluster_centers_

    def diversity(idx, emb):
        centroid = centroids[labels[idx]]
        cos = np.dot(emb, centroid) / (np.linalg.norm(emb) * np.linalg.norm(centroid) + eps)
        return 1 - cos

    df["D"] = [diversity(i, emb) for i, emb in enumerate(X_emb)]
    df["D_norm"] = norm(df["D"])

    # populatiry func
    α, β, γ, η, ζ = 0.35, 0.30, 0.20, 0.10, 0.0500000000000001

    df["popularity_score"] = (
        α * df["P_e_norm"] +
        β * df["R_norm"] +
        γ * df["U_norm"] +
        η * (1 - df["N0_norm"]) + 
        ζ * df["D_norm"]
    )
    print("populatiry func computed")
    df["cluster"] = labels
    return df, X_emb, kmeans

data_pop, embeddings, kmeans_model = compute_popularity(data)

# Detemine the best thresholds based on Gaussian Mixture
scores = data_pop["popularity_score"].values.reshape(-1, 1)

# Fit GMM
gmm = GaussianMixture(n_components=3, random_state=42).fit(scores)
labels = gmm.predict(scores)
probs = gmm.predict_proba(scores)
means = gmm.means_.flatten()
std_devs = np.sqrt(gmm.covariances_.flatten())
weights = gmm.weights_.flatten()

# Sort components by mean
order = np.argsort(means)
means = means[order]
std_devs = std_devs[order]
weights = weights[order]

thresholds = np.sort(gmm.means_.flatten())
print("derived thresholds:", thresholds)

data.loc[:, 'popularity'] = data_pop['popularity_score'].values

threshold_low = (means[0] + means[1]) / 2   # midpoint between first and second mean
threshold_high = (means[1] + means[2]) / 2  # midpoint between second and third mean

print("Thresholds for categories:", threshold_low, threshold_high)

def gmm_category(score):
    if score < threshold_low:
        return "unpopular"
    elif score < threshold_high:
        return "popular"
    else:
        return "very popular"
    
data["popularity_category"] = data["popularity"].apply(gmm_category)
data["popularity_category"].value_counts()

TF-IDF vectorization
Dimensionality reduction
12057 Embdedd.
populatiry func computed
derived thresholds: [0.11470199 0.13951313 0.26274428]
Thresholds for categories: 0.12710756378945923 0.20112870834087676


popularity_category
popular         6286
unpopular       5429
very popular     342
Name: count, dtype: int64

In [6]:
data

,month_number,year_number,bundle_id,bundle_name,bundle_type,price,usage_type,service_class_category,total_rev,total_subscriptions,total_sessions,unique_users,season,configured_volume_mb,configured_volume_min,configured_volume_sms,validity,validity_hours,popularity,popularity_category
0,1,2025,FVX7D840FV3,Maxivoice 90Mins 7 jour@840F,BUNDLE_VOICE,840.0,CHARGED,PREPAID,94717559.86,112759,112759,33989,Winter,0.0,90.0,0.0,7 days,168.0,0.180817,popular
2,1,2025,36848,Zone A and B NDAKO 550GB 30 Jours@26500F,BUNDLE_DATA,26500.0,CHARGED,PREPAID,2782500.00,105,105,105,Winter,563200.0,0.0,0.0,30 days,720.0,0.114912,unpopular
3,1,2025,FVX1D120F,"Maxivoice 7,5Mins and 1,5Mins 1 jour@120F",BUNDLE_VOICE,120.0,CHARGED,HYBRID,480.00,4,4,1,Winter,0.0,9.0,0.0,1 days,24.0,0.107339,unpopular
5,1,2025,36206,Forfait Classic 7 jours 375MB@1200,BUNDLE_DATA,1200.0,CHARGED,HYBRID,22800.00,19,19,12,Winter,375.0,0.0,0.0,7 days,168.0,0.130772,popular
6,1,2025,ALLNET10MN1D450F,Forfait MagicVoice tous reseaux 1 jour 10 mins...,BUNDLE_VOICE,450.0,CHARGED,HYBRID,1350.00,3,3,3,Winter,0.0,0.0,0.0,1 days,24.0,0.115064,unpopular
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12307,10,2025,36207,Forfait Classic 7 jours 750MB@2000,BUNDLE_DATA,2000.0,CHARGED,PREPAID,70000.00,35,35,21,Autumn,750.0,0.0,0.0,7 days,168.0,0.137612,popular
12308,10,2025,36972,MagicNet 20.2GB 30 Jours@13000F,BUNDLE_DATA,13000.0,CHARGED,HYBRID,364000.00,28,28,28,Autumn,20684.8,0.0,0.0,30 days,720.0,0.139095,popular
12309,10,2025,35074,Forfait 1 jour 28 MB,BUNDLE_DATA,130.0,CHARGED,PREPAID,1604070.00,13426,13473,8878,Autumn,28.0,0.0,0.0,1 days,24.0,0.144675,popular
12310,10,2025,80141,Forfait Chrono voice 40 Mins 4H@200F,BUNDLE_VOICE,200.0,CHARGED,PREPAID,4375517.00,22162,22795,10990,Autumn,0.0,40.0,0.0,4 hours,4.0,0.149787,popular


* The Bass diffusion model, developed by Frank Bass, is a mathematical model that describes how new products get adopted in a population over time. It’s widely used in marketing to forecast sales of new products, especially when historical data is limited or non-existent.


* The Bass model is a compact, mechanistic model for how a new product (or feature) is adopted over time. It assumes two forces drive adoption:

    - Innovation (external influence) people adopt because of mass media, advertising, or intrinsic discovery. Parameter: 
    𝑝.

    - Imitation (internal influence) people adopt because they see others adopt (word-of-mouth). Parameter: 
    𝑞.





In [8]:
from experimentals.bass_diffusion import (
    BassModelParams,
    BassDiffusionModel,
    BassCalibrator,
)



#### References


* [PyMC - Bass Diffusion Model](https://www.pymc-marketing.io/en/stable/notebooks/bass/bass_example.html)

* [Chapter 15 Product Market Forecasting using the Bass Model](https://srdas.github.io/MLBook/productForecastingBassModel.html)

* [he Bass Diffusion Model…Explained! The Most Important Shape of the Streaming Wars](https://entertainmentstrategyguy.com/2019/09/11/the-bass-diffusion-modelexplained-the-most-important-shape-of-the-streaming-wars/)